## Exploring Experiment Results
This notebook allows for exploration of experiment results.

In [1]:
import pandas as pd
from sklearn.metrics import classification_report

In [3]:
# Experiment we want to explore - customise these
TARGET_LABEL = "object_region"
EXPERIMENT_NUMBER = 10
# Use 0 to display every row, or a 1-based row number within the experiment.
ROW_NUMBER = 1

In [4]:
# Generated constants
FOLDER_NAME = f"{TARGET_LABEL}_classify"
RESULTS_PATH = f"../results/{FOLDER_NAME}"
EXPERIMENTS_DF_PATH = f"{RESULTS_PATH}/experiments.parquet"

In [5]:
def get_experiment_rows(df, experiment_number, row_number=0):
    """
    Return the requested 1-based row(s) within an experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Returns:
        pd.DataFrame: The selected row as a one-row DataFrame, or every row for the
            experiment when row_number is 0.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    # Filter the DataFrame for the specified experiment number
    df_experiment = df[df["experiment_number"] == experiment_number]

    # Catch errors for invalid experiment numbers or row numbers
    if df_experiment.empty:
        raise ValueError(f"Experiment number {experiment_number} not found in DataFrame.")
    if row_number < 0:
        raise ValueError("ROW_NUMBER must be 0 or a positive integer.")
    if row_number > len(df_experiment):
        raise ValueError(
            f"Row {row_number} not found in experiment {experiment_number}; "
            f"it contains {len(df_experiment)} row(s)."
        )
    # Return all rows if row_number is 0 or one row if a valid row_number is provided
    if row_number == 0:
        return df_experiment
    return df_experiment.iloc[[row_number - 1]]

In [6]:
def print_experiment_info(df, experiment_number, row_number=0):
    """
    Print the model and data configurations, training history, and evaluation results for a
    given experiment number and optional 1-based row number. A row number of 0
    prints every row.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the information for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        # Print model and data configurations
        print(f"\n---- Row {display_row_number}: {row['run_name']} ----\n")
        print("Train Config:")
        for key, value in row["train_config"].items():
            print(f"  {key}: {value}")
        print("\nData Config:")
        for key, value in row["data_config"].items():
            print(f"  {key}: {value}")

        # Print training history
        history = row["history"]
        print("Training History:")
        for epoch_idx, metrics in enumerate(history):
            print(f"  Epoch {epoch_idx + 1}:")
            for metric_name, metric_value in metrics.items():
                print(f"    {metric_name}: {metric_value}")

        # Print evaluation results
        print("\nStandard Test Results:")
        print(f"  Test Accuracy: {row['test_acc']:.4f}")
        print(f"  Test Loss: {row['test_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_weighted_f1_avg']:.4f}")

        print("\nUnseen Matched Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_matched_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_matched_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_matched_weighted_f1_avg']:.4f}")

        print("\nUnseen Related Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_related_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_related_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_related_weighted_f1_avg']:.4f}")

In [7]:
def print_classification_report(df, experiment_number, row_number=0):
    """
    Print classification reports for the standard test set. A row number of 0
    prints a report for every row in the selected experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Classification Reports for Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the classification report for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        print(f"\n---- Row {display_row_number}: {row['run_name']} ----\n")

        # Generate and print the classification report
        report = classification_report(
            row["test_y_true"], row["test_y_pred"], target_names=row["train_labels"]
        )
        print(report)

In [8]:
# Load the required experiment results from the Parquet file
df = pd.read_parquet(EXPERIMENTS_DF_PATH)
df

,experiment_number,experiment_name,run_name,seed,deterministic,freeze_backbone,pretrained_checkpoint,train_config,data_config,model_type,...,test_unseen_matched_weighted_f1_avg,test_unseen_matched_y_true,test_unseen_matched_y_expected,test_unseen_matched_y_pred,test_unseen_related_acc,test_unseen_related_loss,test_unseen_related_weighted_f1_avg,test_unseen_related_y_true,test_unseen_related_y_expected,test_unseen_related_y_pred
0,1,finetuned_comparison,baseline,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",baseline,...,0.255694,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...",15.166667,9.572594,0.192481,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[26, 26, 26, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,1,finetuned_comparison,resnet18,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.304298,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...",17.375000,5.806819,0.223583,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 0, 15..."
2,1,finetuned_comparison,efficientnet_b0,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",efficientnet_b0,...,0.276831,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...",15.166667,6.913476,0.204974,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[26, 26, 26, 26, 26, 26, 27, 15, 27, 27, 27, 1..."
3,1,finetuned_comparison,vit_b_16,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",vit_b_16,...,0.246928,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...",17.000000,8.484512,0.210468,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[26, 26, 26, 26, 27, 27, 0, 0, 27, 27, 27, 27,..."
4,1,finetuned_comparison,deit_tiny,129,True,False,timm/deit_tiny_patch16_224.fb_in1k,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",deit_tiny,...,0.236219,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...",13.277778,8.426949,0.176738,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[26, 26, 26, 26, 28, 27, 26, 27, 27, 27, 27, 2..."
5,1,finetuned_comparison,t3_tiny,129,True,False,alanz-mit/FoundationTactile@e1a16123575eb26e78...,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",t3_tiny,...,0.162010,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...",9.541667,5.739791,0.119675,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[26, 26, 26, 27, 26, 27, 15, 26, 15, 15, 15, 1..."
6,2,frozen_comparison,resnet18,129,True,True,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.218408,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...","[32, 32, 19, 29, 32, 29, 29, 29, 14, 32, 29, 2...",16.486111,3.871577,0.227267,"[0, 0, 0, 0, 0, 0,

In [9]:
print_experiment_info(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Experiment Number: 10
Experiment Name: resnet18_candidate_seed_129_sweep


---- Row 1: pad_none ----

Train Config:
  checkpoint_dir: checkpoints/object_region_classify/010
  early_stopping_min_delta: 0.1
  early_stopping_patience: 3.0
  learning_rate: 2e-05
  min_learning_rate: 1e-06
  model_title: pad_none
  momentum: 0.9
  num_epochs: 25
  optimizer: adamw
  warmup_epochs: 1
  warmup_start_factor: 0.1
  weight_decay: 0.02

Data Config:
  batch_size: 32
  bg_path: data/baseline.jpg
  filtered_force_level: None
  filtered_motion: None
  norm_cache_path: configs/norm_cache.json
  norm_type: dataset
  num_workers: 4
  random_state: 129
  shuffle_map: {'test': False, 'train': True, 'val': False}
  split_size: 0.2
  stratify_label: object_region
  train_augmentations: {'color_jitter': None, 'horizontal_flip': None, 'random_resized_crop': False}
  transform_name: pad_224
Training History:
  Epoch 1:
    epoch: 1
    learning_rate: 2e-05
    train_acc: 66.22415219189412
    train_loss: 1.42

In [10]:
print_classification_report(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Classification Reports for Experiment Number: 10
Experiment Name: resnet18_candidate_seed_129_sweep


---- Row 1: pad_none ----

                             precision    recall  f1-score   support

            football_panels       1.00      1.00      1.00       360
             football_seams       0.99      0.99      0.99       360
              hammer_handle       1.00      0.96      0.98       360
                hammer_head       0.98      0.89      0.93       360
                hammer_neck       0.99      0.99      0.99       360
                   mug_body       0.99      0.76      0.86       360
                 mug_handle       0.68      0.78      0.73       360
                    mug_rim       0.76      0.87      0.81       360
        plastic_bottle_base       0.96      0.89      0.93       360
        plastic_bottle_body       0.93      0.99      0.96       360
         plastic_bottle_cap       0.97      0.97      0.97       360
            scissors_blades       0.90    